In [ ]:
import pandas as pd
from pathlib import Path

# ── Config ────────────────────────────────────────────────────────────────────
# Baseline model name (or full path). Prefix with 'baseline:' to pull from
# results/whisper_baseline/whisper_baseline.csv
BASELINE = 'baseline:whisper'

# Comparison model names (without .csv) or full paths — one or more.
COMPARISON_MODELS = [
    # 'bridge_position_eps_1.5',
    'bridge_dtw_eps_0.5',
    'bridge_dtw_eps_0.5_odesampling',


    # eps, sigma sweep
    'bridge_dtw_fixed_eps_0.3',
    'bridge_dtw_fixed_eps_0.3_ode_renorm',
    'bridge_dtw_fixed_eps_0.5',
    'bridge_dtw_fixed_eps_0.5_ode',
    'bridge_dtw_fixed_eps_0.5_ode_renorm',
    'bridge_dtw_fixed_eps_1.0',
    'bridge_dtw_fixed_eps_1.0_ode',
    'bridge_dtw_fixed_eps_1.0_ode_renorm',

    # x0, sigma sweep
    'bridge_dtw_fixed_x0_0.0_ode',
    'bridge_dtw_fixed_x0_0.0_ode_renorm',
    'bridge_dtw_fixed_x0_0.3',
    'bridge_dtw_fixed_x0_0.3_ode',
    'bridge_dtw_fixed_x0_0.3_ode_renorm',
    'bridge_dtw_fixed_x0_0.5',
    'bridge_dtw_fixed_x0_0.5_ode',
    'bridge_dtw_fixed_x0_0.5_ode_renorm',
    'bridge_dtw_fixed_x0_1.0',
    'bridge_dtw_fixed_x0_1.0_ode',
    'bridge_dtw_fixed_x0_1.0_ode_renorm',



    # 'bridge_dtw_fixed_x0_spanish_0.0_ode',
    # 'bridge_dtw_fixed_x0_spanish_0.0_renorm',
    # 'bridge_dtw_fixed_x0_spanish',
    # 'bridge_dtw_fixed_x0_spanish_renorm',
    # 'bridge_dtw_fixed_x0_spanish_ode',
    # 'bridge_dtw_fixed_x0_spanish_ode_renorm',
    
]

# Metric that drives win/draw/loss classification and best/worst ranking.
PRIMARY_METRIC = 'utt_wer'

# Columns to show in comparison tables. Use None to show all shared metric columns.
SHOW_COLS = ['utt_wer']

# Number of rows to show in the best-wins / worst-losses tables.
TOP_N = 20

# Optional filters (set to None to skip) — applied to baseline and all comparison models
FILTER_L1 = None       # e.g. 'Hindi'
FILTER_SPEAKER = None  # e.g. 'THV'

# ── Path resolution ───────────────────────────────────────────────────────────
ROOT = Path("/vol/gpudata/tsv22-fyp/accent-robust-asr")
BRIDGE_DIR   = Path(f'{ROOT}/results/bridge_eval')
STEERING_DIR = Path(f'{ROOT}/results/e2_steering')
BASELINE_DIR = Path(f'{ROOT}/results/whisper_baseline')

def resolve(name: str) -> Path:
    if name.startswith('baseline:'):
        return BASELINE_DIR / 'whisper_baseline.csv'
    p = Path(name)
    if p.suffix == '.csv' and p.exists():
        return p
    for d in [BRIDGE_DIR, STEERING_DIR]:
        candidate = d / f'{name}.csv'
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f'Cannot find CSV for {name!r}')

path_baseline = resolve(BASELINE)
print(f'Baseline: {BASELINE} -> {path_baseline}')
for name in COMPARISON_MODELS:
    print(f'  {name} -> {resolve(name)}')

Baseline: baseline:whisper -> /vol/gpudata/tsv22-fyp/accent-robust-asr/results/whisper_baseline/whisper_baseline.csv
  bridge_dtw_eps_0.5 -> /vol/gpudata/tsv22-fyp/accent-robust-asr/results/bridge_eval/bridge_dtw_eps_0.5.csv
  bridge_dtw_eps_0.5_odesampling -> /vol/gpudata/tsv22-fyp/accent-robust-asr/results/bridge_eval/bridge_dtw_eps_0.5_odesampling.csv
  bridge_dtw_fixed_eps_0.3 -> /vol/gpudata/tsv22-fyp/accent-robust-asr/results/bridge_eval/bridge_dtw_fixed_eps_0.3.csv
  bridge_dtw_fixed_eps_0.3_ode_renorm -> /vol/gpudata/tsv22-fyp/accent-robust-asr/results/bridge_eval/bridge_dtw_fixed_eps_0.3_ode_renorm.csv
  bridge_dtw_fixed_eps_0.5 -> /vol/gpudata/tsv22-fyp/accent-robust-asr/results/bridge_eval/bridge_dtw_fixed_eps_0.5.csv
  bridge_dtw_fixed_eps_0.5_ode -> /vol/gpudata/tsv22-fyp/accent-robust-asr/results/bridge_eval/bridge_dtw_fixed_eps_0.5_ode.csv
  bridge_dtw_fixed_eps_0.5_ode_renorm -> /vol/gpudata/tsv22-fyp/accent-robust-asr/results/bridge_eval/bridge_dtw_fixed_eps_0.5_ode_re

In [3]:
NON_METRIC = {'utterance_id', 'speaker', 'l1', 'domain', 'wav_path', 'text',
              'prediction', 'reference_norm', 'prediction_norm', 'speaker_type',
              'bridge_split', '_label'}

def load(path: Path, label: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    # normalise column names across different eval scripts
    renames = {'wer': 'utt_wer', 'mer': 'utt_mer', 'per': 'utt_per',
               'whisper_pred': 'prediction', 'whisper_pred_norm': 'prediction_norm'}
    df = df.rename(columns={k: v for k, v in renames.items() if k in df.columns})
    df['_label'] = label
    return df

def apply_filters(df):
    if FILTER_L1 and 'l1' in df.columns:
        df = df[df['l1'] == FILTER_L1]
    if FILTER_SPEAKER and 'speaker' in df.columns:
        df = df[df['speaker'] == FILTER_SPEAKER]
    return df

def compare(da: pd.DataFrame, model_name: str) -> dict:
    """Join filtered baseline rows `da` against `model_name`'s eval CSV and
    precompute win/draw/loss stats, an L1 breakdown, and best/worst tables.

    Columns from the baseline get a `_baseline` suffix, columns from the
    comparison model get a `_bridge` suffix. `{metric}_delta` is always
    `bridge - baseline`: positive means the comparison model is worse than
    the baseline (higher WER), negative means it's better (lower WER).
    """
    db = apply_filters(load(resolve(model_name), model_name))

    metric_cols_a = [c for c in da.columns if c not in NON_METRIC]
    metric_cols_b = [c for c in db.columns if c not in NON_METRIC]
    shared_metrics = [c for c in metric_cols_a if c in metric_cols_b]
    cols_to_show = SHOW_COLS if SHOW_COLS else shared_metrics

    keep_a = ['utterance_id', 'speaker'] + (['l1'] if 'l1' in da.columns else []) + \
             ['text'] + cols_to_show + \
             (['prediction_norm'] if 'prediction_norm' in da.columns else [])
    keep_b = ['utterance_id', 'speaker'] + cols_to_show + \
             (['prediction_norm'] if 'prediction_norm' in db.columns else [])

    merged = da[keep_a].merge(
        db[[c for c in keep_b if c in db.columns]],
        on=['utterance_id', 'speaker'],
        suffixes=('_baseline', '_bridge'),
        how='inner'
    )

    for c in cols_to_show:
        c_base, c_bridge = f'{c}_baseline', f'{c}_bridge'
        if c_base in merged.columns and c_bridge in merged.columns:
            merged[f'{c}_delta'] = merged[c_bridge] - merged[c_base]

    pm_delta = f'{PRIMARY_METRIC}_delta'
    pm_bridge = f'{PRIMARY_METRIC}_bridge'
    is_win  = merged[pm_delta] < 0
    is_loss = merged[pm_delta] > 0
    is_draw = merged[pm_delta] == 0
    is_zero = merged[pm_bridge] == 0
    winloss_counts = {
        'wins':          int(is_win.sum()),
        'corrected':     int((is_win & is_zero).sum()),
        'improved':      int((is_win & ~is_zero).sum()),
        'draws':         int(is_draw.sum()),
        'draws_zero':    int((is_draw & is_zero).sum()),
        'draws_nonzero': int((is_draw & ~is_zero).sum()),
        'losses':        int(is_loss.sum()),
    }

    l1_table = None
    if 'l1' in merged.columns:
        rows = []
        for l1, g in merged.groupby('l1'):
            row = {'l1': l1, 'n': len(g)}
            for c in cols_to_show:
                c_base, c_bridge = f'{c}_baseline', f'{c}_bridge'
                row[f'{c}_baseline'] = g[c_base].mean() * 100
                row[f'{c}_bridge']   = g[c_bridge].mean() * 100
                row[f'{c}_delta']    = (g[c_bridge].mean() - g[c_base].mean()) * 100
            rows.append(row)
        l1_table = pd.DataFrame(rows).set_index('l1').round(2)

    text_cols   = ['utterance_id', 'speaker'] + (['l1'] if 'l1' in merged.columns else []) + ['text']
    pred_cols   = [c for c in merged.columns if 'prediction_norm' in c]
    metric_disp = [c for c in merged.columns if any(c.startswith(m) for m in cols_to_show)]
    browse_cols = text_cols + pred_cols + metric_disp

    best_wins    = merged.sort_values(pm_delta, ascending=True).head(TOP_N)[browse_cols].reset_index(drop=True)
    worst_losses = merged.sort_values(pm_delta, ascending=False).head(TOP_N)[browse_cols].reset_index(drop=True)
    for col in metric_disp:
        best_wins[col]    = (best_wins[col]    * 100).round(2)
        worst_losses[col] = (worst_losses[col] * 100).round(2)

    return {
        'name': model_name,
        'merged': merged,
        'n': len(merged),
        'cols_to_show': cols_to_show,
        'winloss_counts': winloss_counts,
        'l1_table': l1_table,
        'best_wins': best_wins,
        'worst_losses': worst_losses,
    }

da_baseline = apply_filters(load(path_baseline, BASELINE))
print(f'Baseline {BASELINE!r}: {len(da_baseline)} rows')

results = [compare(da_baseline, name) for name in COMPARISON_MODELS]

print(f'\nComputed {len(results)} comparison(s) against baseline:')
for r in results:
    wl = r['winloss_counts']
    print(f"  {r['name']:35s} n={r['n']:5d}  wins={wl['wins']:4d} (corrected={wl['corrected']:4d}, improved={wl['improved']:4d})"
          f"  draws={wl['draws']:4d} (zero={wl['draws_zero']:4d}, nonzero={wl['draws_nonzero']:4d})  losses={wl['losses']:4d}")

Baseline 'baseline:whisper': 31395 rows



Computed 19 comparison(s) against baseline:
  bridge_dtw_eps_0.5                  n= 7796  wins= 635 (corrected= 180, improved= 455)  draws=5935 (zero=3156, nonzero=2779)  losses=1225
  bridge_dtw_eps_0.5_odesampling      n= 7796  wins= 607 (corrected= 168, improved= 439)  draws=6055 (zero=3188, nonzero=2867)  losses=1133
  bridge_dtw_fixed_eps_0.3            n= 7796  wins= 899 (corrected= 269, improved= 630)  draws=6033 (zero=3235, nonzero=2798)  losses= 863
  bridge_dtw_fixed_eps_0.3_ode_renorm n= 7796  wins= 899 (corrected= 256, improved= 643)  draws=6145 (zero=3273, nonzero=2872)  losses= 751
  bridge_dtw_fixed_eps_0.5            n= 7796  wins= 967 (corrected= 303, improved= 664)  draws=5804 (zero=3194, nonzero=2610)  losses=1024
  bridge_dtw_fixed_eps_0.5_ode        n= 7796  wins= 973 (corrected= 290, improved= 683)  draws=6001 (zero=3249, nonzero=2752)  losses= 821
  bridge_dtw_fixed_eps_0.5_ode_renorm n= 7796  wins= 997 (corrected= 281, improved= 716)  draws=6035 (zero=3267, no

In [4]:
from IPython.display import display, Markdown

pm = PRIMARY_METRIC
raw_rows, pct_rows = [], []
for r in results:
    merged, wl, n = r['merged'], r['winloss_counts'], r['n']
    baseline_mean = merged[f'{pm}_baseline'].mean()
    bridge_mean   = merged[f'{pm}_bridge'].mean()

    raw_rows.append({
        'model': r['name'],
        'n': n,
        f'{pm}_baseline_%': baseline_mean * 100,
        f'{pm}_bridge_%':   bridge_mean   * 100,
        f'{pm}_delta_%':    (bridge_mean - baseline_mean) * 100,
        'wins': wl['wins'],
        'corrected': wl['corrected'],
        'improved': wl['improved'],
        'draws': wl['draws'],
        'draws_zero': wl['draws_zero'],
        'draws_nonzero': wl['draws_nonzero'],
        'losses': wl['losses'],
    })
    pct_rows.append({
        'model': r['name'],
        'win_%':           wl['wins']          / n * 100,
        'corrected_%':     wl['corrected']     / n * 100,
        'improved_%':      wl['improved']      / n * 100,
        'draw_%':          wl['draws']         / n * 100,
        'draws_zero_%':    wl['draws_zero']    / n * 100,
        'draws_nonzero_%': wl['draws_nonzero'] / n * 100,
        'loss_%':          wl['losses']        / n * 100,
    })

overview_raw = pd.DataFrame(raw_rows).set_index('model').round(2)
overview_pct = pd.DataFrame(pct_rows).set_index('model').round(2)

display(Markdown('**Raw counts (WER in %; wins split into corrected [→0 WER] / improved [still nonzero]; '
                 'draws split into zero / nonzero):**'))
display(overview_raw)
display(Markdown('**Percentages (of compared utterances `n`):**'))
display(overview_pct)

**Raw counts (WER in %; wins split into corrected [→0 WER] / improved [still nonzero]; draws split into zero / nonzero):**

,n,utt_wer_baseline_%,utt_wer_bridge_%,utt_wer_delta_%,wins,corrected,improved,draws,draws_zero,draws_nonzero,losses
model,,,,,,,,,,,
bridge_dtw_eps_0.5,7796,15.4,19.15,3.74,635,180,455,5935,3156,2779,1225
bridge_dtw_eps_0.5_odesampling,7796,15.4,18.89,3.49,607,168,439,6055,3188,2867,1133
bridge_dtw_fixed_eps_0.3,7796,15.4,16.56,1.16,899,269,630,6033,3235,2798,863
bridge_dtw_fixed_eps_0.3_ode_renorm,7796,15.4,14.96,-0.44,899,256,643,6145,3273,2872,751
bridge_dtw_fixed_eps_0.5,7796,15.4,22.31,6.90,967,303,664,5804,3194,2610,1024
bridge_dtw_fixed_eps_0.5_ode,7796,15.4,18.54,3.14,973,290,683,6001,3249,2752,821
bridge_dtw_fixed_eps_0.5_ode_renorm,7796,15.4,14.70,-0.70,997,281,716,6035,3267,2768,763
bridge_dtw_fixed_eps_1.0,7796,15.4,23.59,8.19,1142,389,753,5325,3041,2284,1328
bridge_dtw_fixed_eps_1.0_ode,7796,15.4,29.18,13.78,1199,415,784,4947,2838,2109,1649


**Percentages (of compared utterances `n`):**

,win_%,corrected_%,improved_%,draw_%,draws_zero_%,draws_nonzero_%,loss_%
model,,,,,,,
bridge_dtw_eps_0.5,8.15,2.31,5.84,76.13,40.48,35.65,15.71
bridge_dtw_eps_0.5_odesampling,7.79,2.15,5.63,77.67,40.89,36.78,14.53
bridge_dtw_fixed_eps_0.3,11.53,3.45,8.08,77.39,41.50,35.89,11.07
bridge_dtw_fixed_eps_0.3_ode_renorm,11.53,3.28,8.25,78.82,41.98,36.84,9.63
bridge_dtw_fixed_eps_0.5,12.40,3.89,8.52,74.45,40.97,33.48,13.13
bridge_dtw_fixed_eps_0.5_ode,12.48,3.72,8.76,76.98,41.68,35.30,10.53
bridge_dtw_fixed_eps_0.5_ode_renorm,12.79,3.60,9.18,77.41,41.91,35.51,9.79
bridge_dtw_fixed_eps_1.0,14.65,4.99,9.66,68.30,39.01,29.30,17.03
bridge_dtw_fixed_eps_1.0_ode,15.38,5.32,10.06,63.46,36.40,27.05,21.15


In [5]:
import jiwer
from IPython.display import display, Markdown

def _cwer(refs, preds):
    return float(jiwer.wer([str(r) for r in refs], [str(p) for p in preds]))

def _cmer(refs, preds):
    return float(jiwer.mer([str(r) for r in refs], [str(p) for p in preds]))

corpus_rows, l1_rows = [], []
for r in results:
    bridge_df = pd.read_csv(resolve(r['name'])).rename(columns={'wer': 'utt_wer'})
    m = r['merged'][['utterance_id', 'speaker', 'l1']].merge(
        bridge_df[['utterance_id', 'speaker', 'reference_norm', 'prediction_norm']],
        on=['utterance_id', 'speaker'], how='inner')
    m = m.merge(
        da_baseline[['utterance_id', 'speaker', 'prediction_norm']].rename(
            columns={'prediction_norm': 'pred_base'}),
        on=['utterance_id', 'speaker'], how='inner')

    refs         = m['reference_norm'].fillna('').tolist()
    preds_base   = m['pred_base'].fillna('').tolist()
    preds_bridge = m['prediction_norm'].fillna('').tolist()

    corpus_rows.append({
        'model':            r['name'],
        'wer_baseline_%':   _cwer(refs, preds_base)   * 100,
        'wer_bridge_%':     _cwer(refs, preds_bridge) * 100,
        'wer_delta_%':      (_cwer(refs, preds_bridge) - _cwer(refs, preds_base)) * 100,
        'mer_baseline_%':   _cmer(refs, preds_base)   * 100,
        'mer_bridge_%':     _cmer(refs, preds_bridge) * 100,
        'mer_delta_%':      (_cmer(refs, preds_bridge) - _cmer(refs, preds_base)) * 100,
    })

    for l1, g in m.groupby('l1'):
        refs_l1   = g['reference_norm'].fillna('').tolist()
        base_l1   = g['pred_base'].fillna('').tolist()
        bridge_l1 = g['prediction_norm'].fillna('').tolist()
        l1_rows.append({
            'model':          r['name'],
            'l1':             l1,
            'wer_baseline_%': _cwer(refs_l1, base_l1)   * 100,
            'wer_bridge_%':   _cwer(refs_l1, bridge_l1) * 100,
            'wer_delta_%':    (_cwer(refs_l1, bridge_l1) - _cwer(refs_l1, base_l1)) * 100,
            'mer_baseline_%': _cmer(refs_l1, base_l1)   * 100,
            'mer_bridge_%':   _cmer(refs_l1, bridge_l1) * 100,
            'mer_delta_%':    (_cmer(refs_l1, bridge_l1) - _cmer(refs_l1, base_l1)) * 100,
        })

display(Markdown('### Corpus WER + MER % (jiwer — total errors / total reference words)'))
display(pd.DataFrame(corpus_rows).set_index('model').round(2))

l1_df = pd.DataFrame(l1_rows)
baseline_wer = (l1_df[l1_df['model'] == l1_df['model'].iloc[0]]
                .set_index('l1')['wer_baseline_%'].rename('baseline'))
baseline_mer = (l1_df[l1_df['model'] == l1_df['model'].iloc[0]]
                .set_index('l1')['mer_baseline_%'].rename('baseline'))

for metric, bl in [('wer', baseline_wer), ('mer', baseline_mer)]:
    display(Markdown(f'### Per-L1 corpus {metric.upper()} % (bridge; baseline row for reference)'))
    pivot = l1_df.pivot(index='model', columns='l1', values=f'{metric}_bridge_%').round(2)
    display(pd.concat([bl.to_frame().T.rename(index={'baseline': 'baseline'}), pivot]))

    display(Markdown(f'### Per-L1 corpus {metric.upper()} % delta (bridge − baseline; negative = improvement)'))
    display(l1_df.pivot(index='model', columns='l1', values=f'{metric}_delta_%').round(2))

### Corpus WER + MER % (jiwer — total errors / total reference words)

,wer_baseline_%,wer_bridge_%,wer_delta_%,mer_baseline_%,mer_bridge_%,mer_delta_%
model,,,,,,
bridge_dtw_eps_0.5,14.71,19.10,4.39,14.5,18.34,3.84
bridge_dtw_eps_0.5_odesampling,14.71,18.85,4.14,14.5,18.11,3.60
bridge_dtw_fixed_eps_0.3,14.71,15.60,0.89,14.5,15.24,0.74
bridge_dtw_fixed_eps_0.3_ode_renorm,14.71,14.40,-0.31,14.5,14.22,-0.29
bridge_dtw_fixed_eps_0.5,14.71,21.86,7.15,14.5,20.18,5.67
bridge_dtw_fixed_eps_0.5_ode,14.71,17.73,3.02,14.5,16.94,2.44
bridge_dtw_fixed_eps_0.5_ode_renorm,14.71,14.17,-0.54,14.5,14.00,-0.50
bridge_dtw_fixed_eps_1.0,14.71,23.47,8.76,14.5,21.41,6.91
bridge_dtw_fixed_eps_1.0_ode,14.71,29.91,15.20,14.5,25.92,11.42


### Per-L1 corpus WER % (bridge; baseline row for reference)

l1,Arabic,Chinese,English,Hindi,Korean,Spanish,Vietnamese
baseline,12.645392,19.19625,3.815341,6.55265,7.901284,22.647421,31.020084
bridge_dtw_eps_0.5,13.590000,30.86000,4.170000,6.83000,8.530000,28.800000,41.950000
bridge_dtw_eps_0.5_odesampling,13.190000,30.54000,4.130000,6.70000,8.320000,28.480000,41.570000
bridge_dtw_fixed_eps_0.3,12.560000,18.24000,3.770000,6.21000,7.940000,22.680000,38.490000
bridge_dtw_fixed_eps_0.3_ode_renorm,12.280000,18.22000,3.730000,6.09000,7.760000,22.230000,31.270000
bridge_dtw_fixed_eps_0.5,16.940000,19.01000,3.850000,6.30000,8.200000,46.960000,54.430000
bridge_dtw_fixed_eps_0.5_ode,12.270000,22.37000,3.630000,6.07000,7.720000,31.220000,42.230000
bridge_dtw_fixed_eps_0.5_ode_renorm,12.020000,17.79000,3.640000,5.99000,7.680000,22.190000,30.730000
bridge_dtw_fixed_eps_1.0,13.140000,30.22000,4.140000,6.88000,8.490000,56.630000,48.390000
bridge_dtw_fixed_eps_1.0_ode,31.400000,39.18000,4.710000,7.23000,13.730000,53.140000,62.370000


### Per-L1 corpus WER % delta (bridge − baseline; negative = improvement)

l1,Arabic,Chinese,English,Hindi,Korean,Spanish,Vietnamese
model,,,,,,,
bridge_dtw_eps_0.5,0.94,11.67,0.35,0.28,0.63,6.15,10.93
bridge_dtw_eps_0.5_odesampling,0.55,11.35,0.31,0.15,0.42,5.83,10.55
bridge_dtw_fixed_eps_0.3,-0.09,-0.96,-0.05,-0.34,0.04,0.03,7.47
bridge_dtw_fixed_eps_0.3_ode_renorm,-0.37,-0.98,-0.09,-0.47,-0.14,-0.41,0.25
bridge_dtw_fixed_eps_0.5,4.29,-0.19,0.03,-0.25,0.30,24.31,23.41
bridge_dtw_fixed_eps_0.5_ode,-0.38,3.17,-0.19,-0.49,-0.18,8.57,11.21
bridge_dtw_fixed_eps_0.5_ode_renorm,-0.63,-1.41,-0.18,-0.57,-0.22,-0.46,-0.29
bridge_dtw_fixed_eps_1.0,0.50,11.02,0.32,0.33,0.59,33.98,17.37
bridge_dtw_fixed_eps_1.0_ode,18.76,19.98,0.89,0.68,5.83,30.49,31.35


### Per-L1 corpus MER % (bridge; baseline row for reference)

l1,Arabic,Chinese,English,Hindi,Korean,Spanish,Vietnamese
baseline,12.491407,18.889216,3.788783,6.519588,7.851281,22.110553,30.127462
bridge_dtw_eps_0.5,13.410000,28.150000,4.140000,6.800000,8.470000,27.440000,37.640000
bridge_dtw_eps_0.5_odesampling,13.030000,27.860000,4.100000,6.670000,8.270000,27.150000,37.370000
bridge_dtw_fixed_eps_0.3,12.400000,17.960000,3.740000,6.190000,7.900000,22.140000,35.090000
bridge_dtw_fixed_eps_0.3_ode_renorm,12.140000,17.970000,3.700000,6.070000,7.720000,21.730000,30.450000
bridge_dtw_fixed_eps_0.5,16.060000,18.690000,3.820000,6.280000,8.160000,37.170000,43.380000
bridge_dtw_fixed_eps_0.5_ode,12.130000,21.160000,3.600000,6.050000,7.690000,28.100000,37.130000
bridge_dtw_fixed_eps_0.5_ode_renorm,11.890000,17.550000,3.610000,5.970000,7.640000,21.700000,29.960000
bridge_dtw_fixed_eps_1.0,12.980000,26.830000,4.110000,6.850000,8.440000,41.900000,40.480000
bridge_dtw_fixed_eps_1.0_ode,26.510000,32.400000,4.680000,7.200000,13.100000,40.500000,46.960000


### Per-L1 corpus MER % delta (bridge − baseline; negative = improvement)

l1,Arabic,Chinese,English,Hindi,Korean,Spanish,Vietnamese
model,,,,,,,
bridge_dtw_eps_0.5,0.92,9.27,0.35,0.28,0.62,5.33,7.51
bridge_dtw_eps_0.5_odesampling,0.53,8.97,0.31,0.15,0.42,5.04,7.24
bridge_dtw_fixed_eps_0.3,-0.09,-0.93,-0.05,-0.33,0.05,0.03,4.96
bridge_dtw_fixed_eps_0.3_ode_renorm,-0.36,-0.92,-0.09,-0.45,-0.13,-0.38,0.32
bridge_dtw_fixed_eps_0.5,3.57,-0.20,0.03,-0.24,0.30,15.06,13.26
bridge_dtw_fixed_eps_0.5_ode,-0.37,2.27,-0.18,-0.47,-0.17,5.99,7.00
bridge_dtw_fixed_eps_0.5_ode_renorm,-0.60,-1.34,-0.18,-0.55,-0.21,-0.41,-0.17
bridge_dtw_fixed_eps_1.0,0.49,7.94,0.32,0.34,0.59,19.79,10.35
bridge_dtw_fixed_eps_1.0_ode,14.01,13.51,0.89,0.68,5.25,18.39,16.83


In [8]:
# ── Save tables ───────────────────────────────────────────────────────────────
# Subfolder under results/eval_tables/ — e.g. "all", "spanish", "hindi"
SAVE_SUBFOLDER = "all"

save_dir = ROOT / "results" / "bridge_eval" / "summary_tables" / SAVE_SUBFOLDER
save_dir.mkdir(parents=True, exist_ok=True)

# Win/draw/loss summary
overview_raw.to_csv(save_dir / "summary_raw.csv")
overview_pct.to_csv(save_dir / "summary_pct.csv")

# Corpus WER + MER
corpus_df = pd.DataFrame(corpus_rows).set_index('model').round(2)
corpus_df.to_csv(save_dir / "corpus_wer_mer.csv")

# Per-L1 breakdowns
for metric in ['wer', 'mer']:
    l1_df.pivot(index='model', columns='l1', values=f'{metric}_bridge_%').round(2).to_csv(
        save_dir / f"per_l1_{metric}_pct.csv")
    l1_df.pivot(index='model', columns='l1', values=f'{metric}_delta_%').round(2).to_csv(
        save_dir / f"per_l1_{metric}_delta.csv")

print(f"Saved to {save_dir}/")
for f in sorted(save_dir.glob("*.csv")):
    print(f"  {f.name}")

Saved to /vol/gpudata/tsv22-fyp/accent-robust-asr/results/bridge_eval/summary_tables/all/
  corpus_wer_mer.csv
  per_l1_mer_delta.csv
  per_l1_mer_pct.csv
  per_l1_wer_delta.csv
  per_l1_wer_pct.csv
  summary_pct.csv
  summary_raw.csv


In [ ]:
from IPython.display import display, Markdown

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 200)

for r in results:
    display(Markdown(f"## {r['name']}"))

    wl, n = r['winloss_counts'], r['n']
    print(f"Utterances compared: {n}")
    print(f"  wins   (model better than baseline): {wl['wins']:5d}  ({wl['wins']/n*100:5.2f}%)")
    print(f"  draws  (tied):                       {wl['draws']:5d}  ({wl['draws']/n*100:5.2f}%)")
    print(f"  losses (model worse than baseline):  {wl['losses']:5d}  ({wl['losses']/n*100:5.2f}%)")

    if r['l1_table'] is not None:
        display(Markdown("**By L1:**"))
        display(r['l1_table'])

    display(Markdown(f"**Best wins — top {TOP_N} by `{PRIMARY_METRIC}` delta (model beats baseline most):**"))
    display(r['best_wins'])

    display(Markdown(f"**Worst losses — top {TOP_N} by `{PRIMARY_METRIC}` delta (model loses to baseline most):**"))
    display(r['worst_losses'])

## bridge_dtw_eps_0.5

Utterances compared: 7796
  wins   (model better than baseline):   635  ( 8.15%)
  draws  (tied):                        5935  (76.13%)
  losses (model worse than baseline):   1225  (15.71%)


**By L1:**

,n,utt_wer_baseline,utt_wer_bridge,utt_wer_delta
l1,,,,
Arabic,1132,0.1308,0.1399,0.0091
Chinese,1130,0.2010,0.2965,0.0955
English,1132,0.0402,0.0448,0.0046
Hindi,1132,0.0708,0.0738,0.0030
Korean,1131,0.0855,0.0902,0.0047
Spanish,1007,0.2364,0.2838,0.0474
Vietnamese,1132,0.3223,0.4214,0.0991


**Best wins — top 20 by `utt_wer` delta (model beats baseline most):**

...

,n,utt_wer_baseline,utt_wer_bridge,utt_wer_delta
l1,,,,
Arabic,1132,0.1308,0.1284,-0.0024
Chinese,1130,0.2010,0.1838,-0.0173
English,1132,0.0402,0.0380,-0.0022
Hindi,1132,0.0708,0.0631,-0.0077
Korean,1131,0.0855,0.0795,-0.0060
Spanish,1007,0.2364,0.4087,0.1723
Vietnamese,1132,0.3223,0.4112,0.0889


**Best wins — top 20 by `utt_wer` delta (model beats baseline most):**

,utterance_id,speaker,l1,text,prediction_norm_baseline,prediction_norm_bridge,utt_wer_baseline,utt_wer_bridge,utt_wer_delta
0,arctic_a0381,SVBI,Hindi,My name's Ferguson,my name is fargusson,my names ferguson,1.000000,0.000000,-1.000000
1,arctic_b0425,BDL,English,"There were orange-green, gold-green, and a copper-green.",there were orange green gold green and a copper green,there were orangegreen goldgreen and a coppergreen,0.857143,0.000000,-0.857143
2,arctic_a0119,HQTV,Vietnamese,Jeanne was turning the bow shoreward,genie would turn in the bull straw world,jeanne was turning the bow straw war,1.166667,0.333333,-0.833333
3,arctic_b0285,BWC,Chinese,The hyena proceeded to dine,high enough to proceed to dying,the hyena proceeded to die,1.000000,0.200000,-0.800000
4,arctic_a0589,BWC,Chinese,I was sick once typhoid,i will seek once time for it,i was sick once typhoic,1.000000,0.200000,-0.800000
5,arctic_b0536,BWC,Chinese,Typhoid did I tell you,what type of fight did i tell you,typhoid did i tell you,0.800000,0.000000,-0.800000
6,arctic_b0207,ZHAA,Arabic,I'm as good as a man she urged,amasgut ezaman she urged,im as good as a man she urged,0.750000,0.000000,-0.750000
7,arctic_b0268,HJK,Korean,Saxon nodded and the boy frowned,sex and knotted and a boyfriend frowned,saxon knotted and the boy frowned,0.833333,0.166667,-0.666667
8,arctic_b0299,HQTV,Vietnamese,Miss Brodie's smile was slightly sarcastic,misbroadies my words like this sarcastic,miss brodys smile was slightly sarcastic,0.833333,0.166667,-0.666667
9,arctic_b0257,HQTV,Vietnamese,Tudor surveyed him with withering disgust,to the survey he was with three discussed,to the survey him with withering disgust,1.166667,0.500000,-0.666667


**Worst losses — top 20 by `utt_wer` delta (model loses to baseline most):**

,utterance_id,speaker,l1,text,prediction_norm_baseline,prediction_norm_bridge,utt_wer_baseline,utt_wer_bridge,utt_wer_delta
0,arctic_b0257,EBVS,Spanish,Tudor surveyed him with withering disgust,the two of us will bite him with great disgust,two years should be him with him with him with him with him with him with hi...,1.166667,73.666667,72.500000
1,arctic_b0238,EBVS,Spanish,The very thought of the effort to swim over was nauseating,the very thought to the effort to swim over was negotiating,the very thought to the effort to swim over was never was never was never wa...,0.181818,39.545455,39.363636
2,arctic_b0180,EBVS,Spanish,I I beg pardon he drawled,i beg your pardon he dropped,i i i i i i i i i i i i i i i i i i i i i i i i i i i i i i i i i i i i i i ...,0.500000,36.666667,36.166667
3,arctic_a0305,HQTV,Vietnamese,They had no fixed values to be altered by adjectives and adverbs,i had no fixed value to be on to my objective and i was,i had no fixed value to be on my my my my my my my my my my my my my my my m...,0.666667,36.583333,35.916667
4,arctic_a0139,EBVS,Spanish,He told himself that as he washed himself and groomed his disheveled clothes,he told himself that as he washed himself and groomed his dishshell clothes,he told himself that as he watched himself and groomed his himself his himse...,0.076923,33.307692,33.230769
5,arctic_a0450,HQTV,Vietnamese,No man ate of the seal meat or the oil,nominate of the silmit or the oil,nominate of the seamate of the ode to the seamate of the seamate of the seam...,0.500000,26.400000,25.900000
6,arctic_b0505,HQTV,Vietnamese,How valiantly I went at it that first day,how clearly i went at it that first day,how how how how how how how how how how how how how how how how how how how ...,0.111111,24.555556,24.444444
7,arctic_b0360,HQTV,Vietnamese,This state of mind comes of an undue prominence of the ego,this day of mine comes upon undue prominence of the eagle,this day of my come come come come come come come come come come come come c...,0.416667,18.500000,18.083333
8,arctic_a0086,HJK,Korean,Death had come with terrible suddenness,death had come with terrible suddenness,thats it,0.000000,1.000000,1.000000
9,arctic_b0313,ZHAA,Arabic,The apron string loomed near and he shied like an unbroken colt,the upper and strength loomed near and he shied like an endbroken cult,the upper and the lower part of the body is a little bit more flexible,0.416667,1.166667,0.750000


In [7]:
# ── Look up specific utterances across all models ─────────────────────────────
# Add (utterance_id, speaker) pairs here to see how the baseline and every
# comparison model handled them, side by side.
LOOKUP_UTTERANCES = [
    ('arctic_a0484', 'BDL'),
    ('arctic_b0319', 'HQTV'),
]

lookup_keys = pd.DataFrame(LOOKUP_UTTERANCES, columns=['utterance_id', 'speaker'])

base_cols = ['utterance_id', 'speaker'] + (['l1'] if 'l1' in da_baseline.columns else []) + \
            ['text', 'prediction_norm', PRIMARY_METRIC]
wide = lookup_keys.merge(da_baseline[base_cols], on=['utterance_id', 'speaker'], how='left')
wide = wide.rename(columns={
    'prediction_norm': f'prediction_norm_{BASELINE}',
    PRIMARY_METRIC: f'{PRIMARY_METRIC}_{BASELINE}',
})

for r in results:
    sub = r['merged'][['utterance_id', 'speaker', 'prediction_norm_bridge', f'{PRIMARY_METRIC}_bridge']].rename(columns={
        'prediction_norm_bridge': f'prediction_norm_{r["name"]}',
        f'{PRIMARY_METRIC}_bridge': f'{PRIMARY_METRIC}_{r["name"]}',
    })
    wide = wide.merge(sub, on=['utterance_id', 'speaker'], how='left')

wide

,utterance_id,speaker,l1,text,prediction_norm_baseline:whisper,utt_wer_baseline:whisper,prediction_norm_bridge_dtw_eps_0.5,utt_wer_bridge_dtw_eps_0.5,prediction_norm_bridge_dtw_eps_0.5_odesampling,utt_wer_bridge_dtw_eps_0.5_odesampling,...,prediction_norm_bridge_dtw_fixed_x0_0.5_ode,utt_wer_bridge_dtw_fixed_x0_0.5_ode,prediction_norm_bridge_dtw_fixed_x0_0.5_ode_renorm,utt_wer_bridge_dtw_fixed_x0_0.5_ode_renorm,prediction_norm_bridge_dtw_fixed_x0_1.0,utt_wer_bridge_dtw_fixed_x0_1.0,prediction_norm_bridge_dtw_fixed_x0_1.0_ode,utt_wer_bridge_dtw_fixed_x0_1.0_ode,prediction_norm_bridge_dtw_fixed_x0_1.0_ode_renorm,utt_wer_bridge_dtw_fixed_x0_1.0_ode_renorm
0,arctic_a0484,BDL,English,No-sir-ee.,no surrey,2.0,no sirree,2.0,no sirree,2.0,...,no surrey,2.0,no surrey,2.0,no sirree,2.0,no surrey,2.0,no surrey,2.0
1,arctic_b0319,HQTV,Vietnamese,Daylight was tired profoundly tired,they lied to a tire a foully tire,1.6,they lied were tired profoundly tired,0.6,they lied were tired were profoundly tired,0.8,...,they lied were tied refowlied tied,1.2,they lied were tied refowlly tied,1.2,they light with tide a firefly,1.2,they light with tide refowlly tide,1.2,they light with tide refowlly tide,1.2
